# Hito 2 - IRB 120: IK de Pieper, LSPB, Jacobiana, dinámica y singularidad (q5 = 0)

**Todo es SIMULACIÓN MATEMÁTICA sobre el URDF aproximado del Hito 1. No se probó nada en el robot real.**

Supuestos que NO vienen del IRB 120 real:
- Los límites `vmax`, `amax` de este notebook son valores de prueba.
- Los parámetros inerciales son `demo_link_params()` (m = 1 kg por link, varilla delgada). Los torques son solo ilustrativos.

Ejecutar con Jupyter desde la carpeta `scripts/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from irb120_fk import forward_kinematics, pose_error
from irb120_ik import ik_pieper
from irb120_traj import LSPB
from irb120_jac import jacobian, singular_values
from irb120_dyn import rne, mass_matrix, torque_for_cartesian_accel
from irb120_dyn_params import demo_link_params
np.set_printoptions(precision=4, suppress=True)
params = demo_link_params()

## 1. IK de Pieper validada con FK

In [ ]:
q_true = np.array([0.4, -0.5, 0.6, 0.7, 0.8, -0.9])
T_des = forward_kinematics(q_true)
sols = ik_pieper(T_des)
print("Soluciones encontradas:", len(sols))
print(f"{'rama':>12} {'err_pos [m]':>12} {'err_ang [rad]':>14}")
for s in sols:
    e_p, e_a = pose_error(forward_kinematics(s['q']), T_des)
    print(f"{str(s['branch']):>12} {e_p:12.2e} {e_a:14.2e}")
wrap = lambda a: np.arctan2(np.sin(a), np.cos(a))
d = [np.linalg.norm(wrap(s['q'] - q_true)) for s in sols]
print("Distancia mínima a q_true:", min(d))

## 2. Trayectoria LSPB (límites de prueba)

In [ ]:
vmax = np.full(6, 2.0)    # rad/s  SUPUESTO
amax = np.full(6, 5.0)    # rad/s^2 SUPUESTO
q0 = np.zeros(6)
qf = min(sols, key=lambda s: np.linalg.norm(wrap(s['q'] - q0)))['q']
qf = q0 + wrap(qf - q0)
lspb = LSPB(q0, qf, vmax, amax)
t, Q, Qd, Qdd = lspb.sample(0.01)
print(f"T = {lspb.T:.3f} s, tb = {lspb.tb:.3f} s")
print("max|qd| =", np.abs(Qd).max(0), " max|qdd| =", np.abs(Qdd).max(0))
fig, ax = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
for k, (Y, n) in enumerate([(Q, 'q [rad]'), (Qd, 'qd [rad/s]'), (Qdd, 'qdd [rad/s²]')]):
    ax[k].plot(t, Y); ax[k].set_ylabel(n); ax[k].grid(True)
ax[2].set_xlabel('t [s]'); ax[0].set_title('LSPB sincronizado')
plt.tight_layout(); plt.show()

## 3. Jacobiana y valores singulares

In [ ]:
J = jacobian(q_true)
print("J =\n", J)
print("Valores singulares:", singular_values(J))
print("det J =", np.linalg.det(J))

## 4. Dinámica inversa (parámetros DEMO)

In [ ]:
tau = np.array([rne(Q[i], Qd[i], Qdd[i], params) for i in range(len(t))])
plt.figure(figsize=(8, 4)); plt.plot(t, tau); plt.grid(True)
plt.xlabel('t [s]'); plt.ylabel('tau [N m]'); plt.title('Torques a lo largo de la LSPB (parámetros DEMO)')
plt.show()

## 5. Singularidad en q5 = 0
Con Euler ZYZ en la muñeca, `qd6 ~ 1/sin(q5)` y `qd4 ~ cot(q5)`. Se barre q5 → 0 con un twist cartesiano fijo y se mide la norma de `qd = J⁻¹ ξ` y de los torques para una aceleración cartesiana fija. En q5 = 0 exacto J no es invertible, por eso se usan valores pequeños.

In [ ]:
q5s = np.logspace(-0.5, -5, 40)
xi = np.array([0.05, 0.0, 0.0, 0.2, 0.1, 0.1])        # twist deseado (supuesto)
a_c = np.array([0.1, 0.0, 0.0, 0.5, 0.3, 0.2])         # aceleración cartesiana deseada (supuesto)
sig_min, qd_n, tau_n = [], [], []
for q5 in q5s:
    q = np.array([0.3, -0.4, 0.5, 0.3, q5, 0.2])
    J = jacobian(q)
    sig_min.append(singular_values(J)[-1])
    qd = np.linalg.solve(J, xi)
    qd_n.append(np.linalg.norm(qd))
    tau, _ = torque_for_cartesian_accel(q, np.zeros(6), a_c, params)
    tau_n.append(np.linalg.norm(tau))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].loglog(q5s, sig_min); ax[0].set_title('sigma_min(J)')
ax[1].loglog(q5s, qd_n, label='|qd|'); ax[1].loglog(q5s, qd_n[-1]*q5s[-1]/np.sin(q5s), '--', label='~1/sin(q5)'); ax[1].legend(); ax[1].set_title('Velocidades articulares')
ax[2].loglog(q5s, tau_n); ax[2].set_title('|tau| (DEMO)')
for a in ax: a.set_xlabel('q5 [rad]'); a.grid(True, which='both')
plt.tight_layout(); plt.show()
print("Pendiente log-log de |qd| vs q5:", np.polyfit(np.log(q5s[-15:]), np.log(qd_n[-15:]), 1)[0])

## Conclusiones
- La IK de Pieper devuelve hasta 8 ramas y cada una reproduce la pose por FK.
- LSPB respeta los límites supuestos y sincroniza las 6 juntas.
- Al acercarse a q5 = 0, sigma_min(J) → 0 y velocidades y torques crecen ~ 1/sin(q5) (pendiente ≈ -1 en log-log).
- Limitaciones: URDF aproximado, límites e inercias de demostración. Resultados simulados, no medidos en el IRB 120 real.